In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

import pandas as pd
    
sys.path.insert(1, "../../")

import precision_mapping as pm
import CAP_tools


sys.path.insert(1, "../../network_control")
import surface_mapping as sfm

# HCP PM Pipeline 

### Parameters

In [2]:
overwrite = False

exclude_subcortex = True
mask = False
sparsity = 0.1
max_trs = 300

silent = True

pm.utils.printer.silent = silent

### paths

In [3]:
results_dir = "/data/data7/network_control/results/HCP_7T_analysis/precision_maps/rfMRI_REST_7T_ALL"
dtseries_list_txt = "/data/data7/network_control/scripts/_params/HCP_7T_CAP_analysis/HCP_7T_CAP_analysis_rfMRI_REST_7T_ALL_dtseries_list.txt"

with open(dtseries_list_txt) as file:
    dtseries_paths = file.read().strip().split()

cached_dtseries_paths = pm.utils.cache_tmp_path(dtseries_paths, write_cache=False)

subjects = [path.split("/MNINon")[0].split("/")[-1] for path in dtseries_paths]

voxel_FC_file_ending = "voxel_FC.npz"
parcel_file_ending = "parcellation.npy"
network_file_ending = "networks.npy"

Caching paths in ~/_tmp:   0%|          | 0/177 [00:00<?, ?it/s]

In [4]:
# import nibabel as nb
# import numpy as np

# from tqdm.auto import tqdm

# def cifti_to_npy(cifti_path, npy_path, dtype="float16"):
#     """ """
#     data = nb.load(cifti_path).get_fdata(caching="unchanged", dtype=dtype)
#     np.save(npy_path, data)

# tmp_files = [z for z in cached_dtseries_paths if "_tmp" in z]

# for cifti_path in tqdm(tmp_files):
#     assert ".dtseries.nii" in cifti_path
#     npy_path = cifti_path.replace(".dtseries.nii", ".npy")
#     if os.path.exists(npy_path):
#         continue
#     cifti_to_npy(cifti_path, npy_path, dtype="float16")


# npy_cached_dtseries_paths = [z.replace(".dtseries.nii", ".npy") if "_tmp" in z else z
#                              for z in cached_dtseries_paths]

# npy_subjects = [subject for dt_path, subject in zip(npy_cached_dtseries_paths, subjects) if dt_path.endswith(".npy")]
# npy_cached_dtseries_paths = [dt_path for dt_path in npy_cached_dtseries_paths if dt_path.endswith(".npy")]

# assert len(npy_subjects) == len(npy_cached_dtseries_paths)

##### Sparse dconn creation

In [5]:
import gc
import torch

# TODO: Correlations are fixed, now add other steps in pipelines

In [ ]:
for max_trs in [None, 300, 1200, 2400]:
    generic_save_paths = []
    for subject in subjects:
        label_tag = pm.utils.create_path_tag(subject, sparsity, mask, exclude_subcortex, max_trs=max_trs)
        generic_save_paths.append(f"{results_dir}/{subject}/{label_tag}_{{file_ending}}")
        # print()
        os.makedirs(f"{results_dir}/{subject}", exist_ok=True)

    voxel_FC_save_paths = [path.format(file_ending=voxel_FC_file_ending) for path in generic_save_paths]
    parcel_save_paths = [path.format(file_ending=parcel_file_ending) for path in generic_save_paths]
    network_save_paths = [path.format(file_ending=network_file_ending) for path in generic_save_paths]
    
    # _ = pm.functional_connectivity.generate_correlation_matrix(cached_dtseries_paths, voxel_FC_save_paths,
    #                                                            max_trs=max_trs, overwrite=overwrite, backend="torch", device="mps")

    pm.parcellate.parcel_detection(voxel_FC_save_paths, parcel_save_paths, n_cores=5, n_reps=50, overwrite=overwrite)
    
    # pm.na.assign_networks_batch(cached_dtseries_paths, parcel_save_paths, network_save_paths, overwrite=overwrite)

Running infomap parcel detection:   0%|          | 0/177 [00:00<?, ?it/s]

Running infomap parcel detection:   0%|          | 0/177 [00:00<?, ?it/s]

Running infomap parcel detection:   0%|          | 0/177 [00:00<?, ?it/s]

/data/data7/network_control/results/HCP_7T_analysis/precision_maps/rfMRI_REST_7T_ALL/157336/157336_S1_TR2400_voxel_FC.npz/data/data7/network_control/results/HCP_7T_analysis/precision_maps/rfMRI_REST_7T_ALL/156334/156334_S1_TR2400_voxel_FC.npz

Running infomap parcel detection:   0%|          | 0/177 [00:00<?, ?it/s]

/data/data7/network_control/results/HCP_7T_analysis/precision_maps/rfMRI_REST_7T_ALL/155938/155938_S1_TR2400_voxel_FC.npz/data/data7/network_control/results/HCP_7T_analysis/precision_maps/rfMRI_REST_7T_ALL/158035/158035_S1_TR2400_voxel_FC.npz/data/data7/network_control/results/HCP_7T_analysis/precision_maps/rfMRI_REST_7T_ALL/158136/158136_S1_TR2400_voxel_FC.npz




/data/data7/network_control/results/HCP_7T_analysis/precision_maps/rfMRI_REST_7T_ALL/159239/159239_S1_TR2400_voxel_FC.npz
/data/data7/network_control/results/HCP_7T_analysis/precision_maps/rfMRI_REST_7T_ALL/162935/162935_S1_TR2400_voxel_FC.npz
/data/data7/network_control/results/HCP_7T_analysis/precision_maps/rfMRI_REST_7T_ALL/164131/164131_S1_TR2400_voxel_FC.npz
/data/data7/network_control/results/HCP_7T_analysis/precision_maps/rfMRI_REST_7T_ALL/164636/164636_S1_TR2400_voxel_FC.npz
/data/data7/network_control/results/HCP_7T_analysis/precision_maps/rfMRI_REST_7T_ALL/165436/165436_S1_TR2400_voxel_FC.npz
/data/data7/network_co

# TODO:

- Create network assignment files (one based on FC + spatial, one just spatial, probably dlabels or npy, also save network assignment weights)